# Spello Model Application
### Apply Domain-Trained Spell Correction to McCray Transcripts

This notebook:
1. Loads pre-trained Spello model and metadata
2. Loads McCray dataset for correction
3. Applies conservative OCR spell correction
4. Tracks and logs all corrections
5. Saves corrected dataset and analysis

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import re
import json
import os
from datetime import datetime
from spello.model import SpellCorrectionModel
import warnings
warnings.filterwarnings('ignore')

# Configuration
APPLICATION_DATE = datetime.now().strftime("%Y-%m-%d_%H-%M")

print(f"Spello Application Started: {datetime.now()}")
print(f"Application session: {APPLICATION_DATE}")

In [ ]:
# Model Loading Functions
def load_model_registry():
    """
    Load model registry to find available trained models
    """
    registry_path = "../models/model_registry.json"
    
    if os.path.exists(registry_path):
        with open(registry_path, 'r') as f:
            registry = json.load(f)
        return registry
    else:
        print(f"No model registry found at {registry_path}")
        return {}

def list_available_models():
    """
    List all available trained models with performance metrics
    """
    registry = load_model_registry()
    
    if not registry:
        print("No trained models found. Please run spello_train.ipynb first.")
        return None
    
    print("Available Models:")
    print("=" * 80)
    
    for model_name, info in registry.items():
        performance = info.get('performance', {})
        print(f"Model: {model_name}")
        print(f"  Version: {info.get('version', 'Unknown')}")
        print(f"  Created: {info.get('created', 'Unknown')}")
        print(f"  Exact Match Rate: {performance.get('exact_match_rate', 0):.1%}")
        print(f"  Word Accuracy: {performance.get('word_accuracy', 0):.1%}")
        print()
    
    return registry

def load_latest_model():
    """
    Load the most recent trained model
    """
    registry = load_model_registry()
    
    if not registry:
        raise ValueError("No trained models found. Run spello_train.ipynb first.")
    
    # Find latest model by creation date
    latest_model = max(registry.items(), 
                      key=lambda x: x[1].get('created', '1900-01-01'))
    
    model_name, model_info = latest_model
    model_path = model_info['files']['model_path']
    
    print(f"Loading model: {model_name}")
    print(f"Model path: {model_path}")
    
    # Load the Spello model
    sp_model = SpellCorrectionModel(language='en')
    sp_model.load(model_path)
    
    # Load metadata
    metadata_path = model_info['files']['metadata_path']
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    # Load whitelist
    whitelist_path = model_info['files']['whitelist_path']
    with open(whitelist_path, 'r', encoding='utf-8') as f:
        whitelist = {line.strip() for line in f if line.strip()}
    
    print(f"Model loaded successfully!")
    print(f"Performance: {model_info['performance']['exact_match_rate']:.1%} exact match rate")
    
    return sp_model, metadata, whitelist, model_name

# List available models and load the latest
registry = list_available_models()

if registry:
    model, model_metadata, whitelist, current_model_name = load_latest_model()
    print(f"\nUsing model: {current_model_name}")
    print(f"Whitelist contains {len(whitelist)} protected terms")
else:
    print("Please run spello_train.ipynb first to create a trained model.")

In [ ]:
# Load Dataset for Correction
def load_mccray_dataset():
    """
    Load McCray dataset and identify transcript column to correct
    """
    # Try different possible data files
    possible_files = [
        "../data/mccray/october_sprint/mccray_high_english.csv",
        "../data/mccray/october_sprint/mccray_spello_corrected.csv",  # Previous output
        "../data/mccray/changed_data/McCray+.xlsx"
    ]
    
    dataset_path = None
    for file_path in possible_files:
        if os.path.exists(file_path):
            dataset_path = file_path
            break
    
    if not dataset_path:
        raise FileNotFoundError(f"No McCray dataset found. Tried: {possible_files}")
    
    print(f"Loading dataset: {dataset_path}")
    
    # Load based on file type
    if dataset_path.endswith('.csv'):
        df = pd.read_csv(dataset_path)
    elif dataset_path.endswith('.xlsx'):
        df = pd.read_excel(dataset_path)
    else:
        raise ValueError(f"Unsupported file format: {dataset_path}")
    
    print(f"Loaded {len(df)} documents")
    print(f"Available columns: {list(df.columns)}")
    
    return df, dataset_path

def identify_transcript_column(df):
    """
    Identify which column contains the transcript text to correct
    """
    # Priority order for transcript columns
    possible_columns = [
        'Semi-clean Transcript',
        'Cleaned Transcript', 
        'Original Transcript',
        'Transcript'
    ]
    
    target_column = None
    
    for col in possible_columns:
        if col in df.columns:
            target_column = col
            break
    
    if target_column:
        non_empty = df[target_column].notna().sum()
        total_chars = df[target_column].dropna().astype(str).str.len().sum()
        
        print(f"\nTarget column: '{target_column}'")
        print(f"Non-empty documents: {non_empty}/{len(df)}")
        print(f"Total characters: {total_chars:,}")
        
        # Show sample
        sample_text = df[target_column].dropna().iloc[0] if non_empty > 0 else "No data"
        print(f"Sample text: {str(sample_text)[:200]}...")
        
        return target_column
    else:
        print("\nNo suitable transcript column found!")
        print("Available columns:", list(df.columns))
        return None

# Load dataset
if 'model' in locals():
    df, dataset_file = load_mccray_dataset()
    transcript_column = identify_transcript_column(df)
    
    if not transcript_column:
        print("Cannot proceed without a transcript column to correct.")
else:
    print("Model not loaded. Cannot proceed with dataset loading.")

In [ ]:
# Conservative OCR Correction Functions
def is_likely_proper_noun(word):
    """Check if word is likely a proper noun based on capitalization"""
    return len(word) > 1 and word[0].isupper() and word[1:].islower()

def is_protected_word(word, whitelist):
    """Check if word should be protected from correction"""
    # Exact match
    if word in whitelist:
        return True
    
    # Case-insensitive match
    if word.lower() in [w.lower() for w in whitelist]:
        return True
    
    # Years (1800-2023)
    if re.match(r'^(?:18|19|20)\d{2}$', word):
        return True
    
    # Initials
    if re.match(r'^[A-Z]\.$', word):
        return True
        
    return False

def conservative_spell_correct(text, model, whitelist, confidence_threshold=0.7):
    """
    Apply spell correction with conservative rules for OCR
    Returns corrected text and detailed correction log
    """
    if pd.isna(text) or not str(text).strip():
        return text, []
    
    text = str(text)
    corrections_log = []
    
    try:
        # Tokenize while preserving punctuation and spacing
        tokens = re.findall(r'\b\w+\b|\W+', text)
        corrected_tokens = []
        
        for token in tokens:
            # Skip non-word tokens
            if not re.match(r'\w+', token):
                corrected_tokens.append(token)
                continue
            
            original_token = token
            
            # Apply protection rules
            if is_protected_word(token, whitelist):
                corrected_tokens.append(token)
                corrections_log.append({
                    'original': token,
                    'suggested': token,
                    'type': 'protected',
                    'applied': False,
                    'reason': 'whitelist_protected'
                })
                continue
            
            # Handle proper nouns conservatively
            if is_likely_proper_noun(token):
                try:
                    result = model.spell_correct(token)
                    suggested = result.get('spell_corrected_text', token) if isinstance(result, dict) else str(result)
                    
                    # Only correct proper nouns if suggestion is also capitalized
                    if suggested != token and suggested[0].isupper():
                        corrections_log.append({
                            'original': token,
                            'suggested': suggested,
                            'type': 'proper_noun',
                            'applied': True,
                            'reason': 'proper_noun_corrected'
                        })
                        corrected_tokens.append(suggested)
                    else:
                        corrected_tokens.append(token)
                        if suggested != token:
                            corrections_log.append({
                                'original': token,
                                'suggested': suggested,
                                'type': 'proper_noun',
                                'applied': False,
                                'reason': 'proper_noun_rejected'
                            })
                except Exception:
                    corrected_tokens.append(token)
            else:
                # Regular word correction
                try:
                    result = model.spell_correct(token)
                    suggested = result.get('spell_corrected_text', token) if isinstance(result, dict) else str(result)
                    
                    if suggested != token:
                        corrections_log.append({
                            'original': token,
                            'suggested': suggested,
                            'type': 'regular',
                            'applied': True,
                            'reason': 'regular_corrected'
                        })
                        corrected_tokens.append(suggested)
                    else:
                        corrected_tokens.append(token)
                except Exception:
                    corrected_tokens.append(token)
        
        corrected_text = ''.join(corrected_tokens)
        return corrected_text, corrections_log
        
    except Exception as e:
        print(f"Error in correction: {e}")
        return text, []

# Test the correction function
if 'model' in locals() and 'whitelist' in locals():
    test_text = "Dr. McCiay published the Lighthouse and lnformer newspaper in Charleston, South Caiolina in 1950."
    
    print("Testing correction function:")
    print(f"Original: {test_text}")
    
    corrected, log = conservative_spell_correct(test_text, model, whitelist)
    print(f"Corrected: {corrected}")
    print(f"Corrections made: {len([c for c in log if c['applied']])}")
    
    for correction in log:
        if correction['applied']:
            print(f"  {correction['original']} → {correction['suggested']} ({correction['type']})")
else:
    print("Model not available for testing")

In [ ]:
# Apply Corrections to Dataset
def apply_corrections_to_dataset(df, text_column, model, whitelist, progress_interval=50):
    """
    Apply spell correction to entire dataset with progress tracking
    """
    corrected_texts = []
    all_corrections = []
    processing_stats = {
        'total_documents': len(df),
        'processed': 0,
        'empty_documents': 0,
        'total_corrections': 0,
        'start_time': datetime.now()
    }
    
    print(f"Applying corrections to {len(df)} documents...")
    print(f"Progress will be shown every {progress_interval} documents")
    
    for idx, row in df.iterrows():
        # Progress reporting
        if idx % progress_interval == 0 and idx > 0:
            elapsed = (datetime.now() - processing_stats['start_time']).total_seconds()
            rate = idx / elapsed if elapsed > 0 else 0
            eta = (len(df) - idx) / rate if rate > 0 else 0
            print(f"  Progress: {idx}/{len(df)} ({idx/len(df):.1%}) - Rate: {rate:.1f} docs/sec - ETA: {eta/60:.1f} min")
        
        original_text = row[text_column] if pd.notna(row[text_column]) else ""
        
        if original_text.strip():
            corrected_text, corrections = conservative_spell_correct(
                original_text, model, whitelist
            )
            
            # Add document metadata to each correction
            for correction in corrections:
                correction.update({
                    'document_index': idx,
                    'document_id': row.get('Original Index', idx),
                    'title': row.get('Title', 'Unknown'),
                    'year': row.get('Year', row.get('Date', 'Unknown'))
                })
                all_corrections.append(correction)
            
            processing_stats['total_corrections'] += len([c for c in corrections if c['applied']])
        else:
            corrected_text = original_text
            processing_stats['empty_documents'] += 1
        
        corrected_texts.append(corrected_text)
        processing_stats['processed'] += 1
    
    processing_stats['end_time'] = datetime.now()
    processing_stats['total_time_seconds'] = (
        processing_stats['end_time'] - processing_stats['start_time']
    ).total_seconds()
    
    return corrected_texts, all_corrections, processing_stats

def analyze_corrections(corrections_log):
    """
    Analyze correction patterns and statistics
    """
    if not corrections_log:
        return {}
    
    df_corrections = pd.DataFrame(corrections_log)
    
    analysis = {
        'total_corrections_attempted': len(corrections_log),
        'corrections_applied': len(df_corrections[df_corrections['applied'] == True]),
        'corrections_rejected': len(df_corrections[df_corrections['applied'] == False]),
        'by_type': df_corrections['type'].value_counts().to_dict(),
        'by_reason': df_corrections['reason'].value_counts().to_dict(),
        'most_common_corrections': (
            df_corrections[df_corrections['applied'] == True]
            .groupby(['original', 'suggested'])
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False)
            .head(10)
            .to_dict('records')
        )
    }
    
    return analysis

# Apply corrections if we have everything loaded
if all(var in locals() for var in ['model', 'df', 'transcript_column', 'whitelist']):
    print(f"\n=== APPLYING CORRECTIONS ===")
    print(f"Model: {current_model_name}")
    print(f"Dataset: {len(df)} documents")
    print(f"Target column: {transcript_column}")
    
    # Apply corrections
    corrected_transcripts, correction_log, processing_stats = apply_corrections_to_dataset(
        df, transcript_column, model, whitelist
    )
    
    # Add corrected column to dataframe
    df['Spello Corrected Transcript'] = corrected_transcripts
    
    # Analyze corrections
    correction_analysis = analyze_corrections(correction_log)
    
    print(f"\n=== CORRECTION RESULTS ===")
    print(f"Processing time: {processing_stats['total_time_seconds']:.1f} seconds")
    print(f"Documents processed: {processing_stats['processed']}")
    print(f"Empty documents skipped: {processing_stats['empty_documents']}")
    print(f"Total corrections applied: {correction_analysis.get('corrections_applied', 0)}")
    print(f"Total corrections rejected: {correction_analysis.get('corrections_rejected', 0)}")
    
    print(f"\nCorrections by type:")
    for corr_type, count in correction_analysis.get('by_type', {}).items():
        print(f"  {corr_type}: {count}")
        
else:
    print("Missing required components. Cannot apply corrections.")
    print(f"Available: {[var for var in ['model', 'df', 'transcript_column', 'whitelist'] if var in locals()]}")

In [ ]:
# Save Results and Analysis
def save_correction_results(df, correction_log, correction_analysis, processing_stats, model_name, session_id):
    """
    Save corrected dataset and comprehensive analysis
    """
    output_dir = "../data/mccray/october_sprint"
    os.makedirs(output_dir, exist_ok=True)
    
    # Save corrected dataset
    corrected_dataset_path = os.path.join(output_dir, f"mccray_spello_corrected_{session_id}.csv")
    df.to_csv(corrected_dataset_path, index=False, encoding='utf-8')
    
    # Save corrections log
    if correction_log:
        corrections_df = pd.DataFrame(correction_log)
        corrections_log_path = os.path.join(output_dir, f"spello_corrections_log_{session_id}.csv")
        corrections_df.to_csv(corrections_log_path, index=False, encoding='utf-8')
    else:
        corrections_log_path = None
    
    # Save comprehensive analysis
    analysis_report = {
        'session_info': {
            'session_id': session_id,
            'model_used': model_name,
            'application_date': APPLICATION_DATE,
            'dataset_source': dataset_file
        },
        'processing_statistics': processing_stats,
        'correction_analysis': correction_analysis,
        'model_metadata': model_metadata,
        'output_files': {
            'corrected_dataset': corrected_dataset_path,
            'corrections_log': corrections_log_path
        }
    }
    
    analysis_path = os.path.join(output_dir, f"spello_analysis_report_{session_id}.json")
    with open(analysis_path, 'w', encoding='utf-8') as f:
        json.dump(analysis_report, f, indent=2, ensure_ascii=False, default=str)
    
    return {
        'corrected_dataset': corrected_dataset_path,
        'corrections_log': corrections_log_path,
        'analysis_report': analysis_path
    }

def show_correction_examples(df, original_column, corrected_column, num_examples=5):
    """
    Show examples of corrections made
    """
    print(f"\n=== CORRECTION EXAMPLES ===")
    
    examples_found = 0
    for idx, row in df.iterrows():
        original = str(row[original_column]) if pd.notna(row[original_column]) else ""
        corrected = str(row[corrected_column]) if pd.notna(row[corrected_column]) else ""
        
        if original != corrected and original.strip() and examples_found < num_examples:
            print(f"\nExample {examples_found + 1}:")
            print(f"Document: {row.get('Title', 'Unknown')[:60]}...")
            print(f"Original:  {original[:150]}...")
            print(f"Corrected: {corrected[:150]}...")
            
            # Highlight differences
            orig_words = original.split()[:20]  # First 20 words
            corr_words = corrected.split()[:20]
            
            if len(orig_words) == len(corr_words):
                changes = []
                for ow, cw in zip(orig_words, corr_words):
                    if ow != cw:
                        changes.append(f"{ow}→{cw}")
                
                if changes:
                    print(f"Changes: {', '.join(changes)}")
            
            examples_found += 1
    
    if examples_found == 0:
        print("No examples found with significant changes.")

# Save results if corrections were applied
if 'corrected_transcripts' in locals():
    print(f"\n=== SAVING RESULTS ===")
    
    output_files = save_correction_results(
        df, correction_log, correction_analysis, 
        processing_stats, current_model_name, APPLICATION_DATE
    )
    
    print("Files saved:")
    for file_type, path in output_files.items():
        if path:
            print(f"  {file_type}: {path}")
    
    # Show examples
    show_correction_examples(df, transcript_column, 'Spello Corrected Transcript')
    
    # Show most common corrections
    if correction_analysis.get('most_common_corrections'):
        print(f"\n=== MOST COMMON CORRECTIONS ===")
        for correction in correction_analysis['most_common_corrections'][:10]:
            print(f"  '{correction['original']}' → '{correction['suggested']}' ({correction['count']} times)")

else:
    print("No corrections to save.")

## Application Complete!

### Session Summary:
- **Model Used**: Loaded from model registry
- **Dataset**: McCray historical documents 
- **Corrections Applied**: Conservative OCR spell correction
- **Results**: Saved with comprehensive analysis

### Output Files:
- **Corrected Dataset**: Contains original data plus `Spello Corrected Transcript` column
- **Corrections Log**: Detailed log of every correction attempted and applied
- **Analysis Report**: Comprehensive statistics and metadata

### Key Features:
- ✅ **Conservative approach**: Protected proper nouns and important terms
- ✅ **Audit trail**: Complete tracking of all corrections
- ✅ **Performance metrics**: Processing statistics and analysis
- ✅ **Model versioning**: Linked to specific trained model
- ✅ **Quality examples**: Sample corrections for review

### Next Steps:
1. **Review corrections**: Check examples and common corrections
2. **Quality assessment**: Spot-check corrected transcripts
3. **Model iteration**: Use feedback to improve model if needed
4. **Production use**: Apply to additional datasets as needed

### Configuration Options:
- Adjust confidence thresholds in `conservative_spell_correct()`
- Modify whitelist terms for better protection
- Retrain model with additional domain content if needed